Создадим директорию куда поместим наши текста, модули, и другие промежуточные шаги.

In [16]:
%cd /home/jovyan/work
%mkdir homework1_dir
# ! mv homework1_dwh/bible.txt homework1_dwh/Poseschenia_sai_774_tov.csv /home/jovyan/work/homework1_dir
%cd homework1_dir
! ls -la

/home/jovyan/work
mkdir: cannot create directory ‘homework1_dir’: File exists
/home/jovyan/work/homework1_dir
total 39852
drwxr-xr-x 3 root   root     4096 Sep 21 13:13 .
drwxr-xr-x 7 jovyan 1000     4096 Sep 20 20:12 ..
-rw-r--r-- 1 root   root  4047392 Sep 20 15:42 bible.txt
drwxr-xr-x 2 root   root     4096 Sep 21 07:49 .ipynb_checkpoints
-rw-r--r-- 1 root   root 36443383 Sep 20 18:32 Poseschenia_sai_774_tov.csv
-rw-r--r-- 1 root   root     1493 Sep 28 19:01 top5_local.txt
-rw-r--r-- 1 root   root     3332 Sep 28 19:01 top_5_sites.py
-rw-r--r-- 1 root   root   284955 Sep 28 19:01 wordcount1_local.txt
-rw-r--r-- 1 root   root     1460 Sep 28 19:01 wordcount_hw.py


In [17]:
! hdfs dfsadmin -report | grep -E "Live datanodes|Name:"

Live datanodes (2):
Name: 172.19.0.2:9866 (map-reduce-infra3_datanode1_1.map-reduce-infra3_spark_mts)
Name: 172.19.0.3:9866 (map-reduce-infra3_datanode2_1.map-reduce-infra3_spark_mts)


Скачаем файл с текстом для задания 1

In [18]:
! curl -o homework1_dir/bible.txt "https://psv4.vkuserphoto.ru/s/v1/d2/rUrKEFdoTIpuNOwqHQdxk46hWiICskO4tMmo7JUJ8wiHCioGBHZ0Yo5QZGSpiRKuGvoafX_0FR5fZ07_3MxSfJDVF8Bh0HW4OZ9O6WlzguoBMPZrmwqeFpb3ixU1u7Aoi2WAl1NudTwH/bible.txt"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0Warning: Failed to create the file homework1_dir/bible.txt: No such file or 
100   140  100   140    0     0    447      0 --:--:-- --:--:-- --:--:--   447
curl: (23) Failure writing output to destination


Добавим csv файл для задания 2

In [19]:
! mv Poseschenia_sai_774_tov.csv homework1_dir/
!ls -la

mv: cannot move 'Poseschenia_sai_774_tov.csv' to 'homework1_dir/': Not a directory
total 39852
drwxr-xr-x 3 root   root     4096 Sep 21 13:13 .
drwxr-xr-x 7 jovyan 1000     4096 Sep 20 20:12 ..
-rw-r--r-- 1 root   root  4047392 Sep 20 15:42 bible.txt
drwxr-xr-x 2 root   root     4096 Sep 21 07:49 .ipynb_checkpoints
-rw-r--r-- 1 root   root 36443383 Sep 20 18:32 Poseschenia_sai_774_tov.csv
-rw-r--r-- 1 root   root     1493 Sep 28 19:01 top5_local.txt
-rw-r--r-- 1 root   root     3332 Sep 28 19:01 top_5_sites.py
-rw-r--r-- 1 root   root   284955 Sep 28 19:01 wordcount1_local.txt
-rw-r--r-- 1 root   root     1460 Sep 28 19:01 wordcount_hw.py


Напишем mapper и reducer для задания 1.

In [20]:
%%writefile wordcount_hw.py

import sys
import re

def mapper():
    for i in sys.stdin: # проходимся в цикле по тому, что приходит с потока ввода
        for j in i.split(): # разбиваем строку на слова по пробелам
            if len(j) > 4: # берём только слова длиной больше 4 символов
                print(f'{j}\t1') # выводим слова в поток вывода

def reducer():
    current_word = None # задали текущее слово
    current_count = 0 # задали текущее кол-во слов
    for i in sys.stdin:
        word, count = i.strip().split('\t', 1) # распаковываем входные строки на слово и его число
        count = int(count)
        if word == current_word or current_word is None: # проверяем совпадает ли новое слово с текущим
            current_count +=count
        else: 
            print(f'{current_word}\t{current_count}') # если слово новое то выводим текщее слово 
            # и его кол-во
            current_count =count
        current_word = word
    print(f'{current_word}\t{current_count}') # вывели последнее слово

if __name__=='__main__':
    mode = sys.argv[1]
    if mode == 'mapper':
        mapper()
    elif mode == 'reducer':
        reducer()
    

Overwriting wordcount_hw.py


Для проверки нашего кода запустим его локально при помощи pipeline. 

In [21]:
!ls -la

total 39852
drwxr-xr-x 3 root   root     4096 Sep 21 13:13 .
drwxr-xr-x 7 jovyan 1000     4096 Sep 20 20:12 ..
-rw-r--r-- 1 root   root  4047392 Sep 20 15:42 bible.txt
drwxr-xr-x 2 root   root     4096 Sep 21 07:49 .ipynb_checkpoints
-rw-r--r-- 1 root   root 36443383 Sep 20 18:32 Poseschenia_sai_774_tov.csv
-rw-r--r-- 1 root   root     1493 Sep 28 19:01 top5_local.txt
-rw-r--r-- 1 root   root     3332 Sep 28 19:01 top_5_sites.py
-rw-r--r-- 1 root   root   284955 Sep 28 19:01 wordcount1_local.txt
-rw-r--r-- 1 root   root     1460 Sep 28 19:08 wordcount_hw.py


In [22]:
%%time
! cat bible.txt | python3 wordcount_hw.py mapper | sort -k1,1 | python3 wordcount_hw.py reducer > wordcount1_local.txt

CPU times: user 8.14 ms, sys: 8.58 ms, total: 16.7 ms
Wall time: 870 ms


In [23]:
! head wordcount1_local.txt

Aaron	205
Aaron,	95
Aaron.	11
Aaron:	2
Aaron;	5
Aaron?	1
Aaronites,	2
Aaron's	31
Abaddon,	1
Abagtha,	1


Напишем mapper reducer для задачи 2.

In [24]:
%%writefile top_5_sites.py

import sys
import csv

def mapper():
    reader = csv.reader(sys.stdin, delimiter=';')
    for row in reader:
        try:
            url = row[0].strip()
            date = row[1].strip().split(' ')[0]
            if not url or not date:
                continue
            print(f"{date}+{url}\t1")
        except Exception as e:
            sys.stderr.write(f"error: {e}\n")
 
 
def reducer():
    current_key = None    # (date, url), которую сейчас суммируем
    current_count = 0     # кол-во ключей уже насчитанных
    tracked_date = None   # дата, для которой сейчас копится top-5
 
    top_5_counts = [0] * 5
    top_5_samples = [None] * 5
 
    for line in sys.stdin:
        try:
            key, cnt = line.strip().split('\t')
            date, url = key.split('+')
            count = int(cnt)
        except Exception as err:
            sys.stderr.write(f"error: {err}\n")
            continue
 
        if current_key is None: # самая первая строка, запуск счетчика 
            current_key = (date, url)
            current_count = count
            tracked_date = date
            continue
 
        if (date, url) == current_key: # если ключ совпал с рассматриваемым (текущим) ключом, просто суммируем
            current_count += count
            continue
 
        #  записали предыдущий url, поскольку появился новый ключ
        _, prev_url = current_key
 
        # Поскольку поступил новый url надо обновить топ
        min_count = min(top_5_counts)
        if current_count > min_count:
            ind_min = top_5_counts.index(min_count)
            top_5_counts[ind_min] = current_count
            top_5_samples[ind_min] = prev_url

        if date != tracked_date: # если появился сэмпл с новой датой, то надо опубликовать сформировавшийся топ
            filtered = [(c, s) for c, s in zip(top_5_counts, top_5_samples) if s is not None]
            for c, s in sorted(filtered, key=lambda x: -x[0]):
                print(f"{tracked_date}\t{s}\t{c}")
            top_5_counts = [0] * 5
            top_5_samples = [None] * 5
            tracked_date = date
 
        # посчитали новый ключ
        current_key = (date, url)
        current_count = count
 
    # если в качестве последнего сэмпла придет уже имеющийся ключ, то надо учесть его кол-во в топе
    if current_key is not None:
        _, prev_url = current_key
        min_count = min(top_5_counts)
        if current_count > min_count:
            ind_min = top_5_counts.index(min_count)
            top_5_counts[ind_min] = current_count
            top_5_samples[ind_min] = prev_url
 
        filtered = [(c, s) for c, s in zip(top_5_counts, top_5_samples) if s is not None]
        for c, s in sorted(filtered, key=lambda x: -x[0]):
            print(f"{tracked_date}\t{s}\t{c}")
            
if __name__ == "__main__":
    mode = sys.argv[1]
    if mode == "mapper":
        mapper()
    elif mode == "reducer":
        reducer()

Overwriting top_5_sites.py


Запустим локально mapreduce для второго задания

In [25]:
%%time
! cat Poseschenia_sai_774_tov.csv | python3 top_5_sites.py mapper | sort -k1,1 | python3 top_5_sites.py reducer > top5_local.txt

CPU times: user 30.8 ms, sys: 20.9 ms, total: 51.7 ms
Wall time: 4.49 s


In [26]:
! head -20 top5_local.txt

2024-05-26	https://gonzales-bautista.com/	335
2024-05-26	http://smith.com/	235
2024-05-26	https://www.smith.com/	221
2024-05-26	http://www.smith.com/	212
2024-05-26	https://smith.com/	212
2024-05-27	https://gonzales-bautista.com/	376
2024-05-27	https://www.smith.com/	270
2024-05-27	https://smith.com/	236
2024-05-27	http://smith.com/	215
2024-05-27	http://www.smith.com/	208
2024-05-28	https://gonzales-bautista.com/	368
2024-05-28	https://smith.com/	256
2024-05-28	https://www.smith.com/	251
2024-05-28	http://smith.com/	224
2024-05-28	http://www.smith.com/	204
2024-05-29	https://gonzales-bautista.com/	402
2024-05-29	https://www.smith.com/	242
2024-05-29	http://www.smith.com/	223
2024-05-29	https://smith.com/	220
2024-05-29	http://smith.com/	206


Создадим соответствующие директории в hdfs.

In [27]:
! hdfs dfs -ls /

Found 2 items
drwxr-xr-x   - hadoop users          0 2026-09-28 18:57 /hw1
drwxr-xr-x   - hadoop users          0 2026-09-28 19:00 /tmp


In [28]:
import os # пропишем путь для  java
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.environ["JAVA_HOME"] + "/bin:" + os.environ["PATH"]

In [29]:
! hdfs dfs -mkdir /hw1
! hdfs dfs -mkdir /hw1/wordcount_hw
! hdfs dfs -mkdir /hw1/wordcount_hw/input
! hdfs dfs -put ./bible.txt /hw1/wordcount_hw/input/
! hdfs dfs -du -h /hw1/wordcount_hw/input/
! hdfs dfs -rm -r /hw1/wordcount_hw/output

mkdir: `/hw1': File exists
mkdir: `/hw1/wordcount_hw': File exists
mkdir: `/hw1/wordcount_hw/input': File exists
3.9 M  3.9 M  /hw1/wordcount_hw/input/bible.txt
rm: `/hw1/wordcount_hw/output': No such file or directory


In [30]:
%%bash
# Запуск Hadoop Streaming для word count:
# -D mapreduce.job.name  — имя job'ы
# -D mapreduce.job.reduces — кол-во reducer'ов
# -files — файл, который отправляется на ноды
# -mapper / -reducer — команды запуска mapper'а и reducer'а
# -input / -output — пути к исходным данным и результату в HDFS
hadoop jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar \
-D mapreduce.job.name="home_work_word_count" \
-D mapreduce.job.reduces=3 \
-files wordcount_hw.py \
-mapper "python3 wordcount_hw.py mapper" \
-reducer "python3 wordcount_hw.py reducer" \
-input /hw1/wordcount_hw/input/ \
-output /hw1/wordcount_hw/output/

packageJobJar: [/tmp/hadoop-unjar5242555656547657369/] [] /tmp/streamjob12346064549018836934.jar tmpDir=null
2026-09-28 19:09:35,305 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.19.0.4:8032
2026-09-28 19:09:35,392 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.19.0.4:8032
2026-09-28 19:09:35,545 INFO  [main] mapreduce.JobResourceUploader (JobResourceUploader.java:disableErasureCodingForPath(907)) - Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1790621459909_0003
2026-09-28 19:09:36,306 INFO  [main] mapred.FileInputFormat (FileInputFormat.java:listStatus(267)) - Total input files to process : 1
2026-09-28 19:09:36,360 INFO  [main] mapreduce.JobSubmitter (JobSubmitter.java:submitJobInternal(203)) - number of splits:2
2026-09-28 

Проверим выполнение 

In [31]:
! hdfs dfs -ls /hw1/wordcount_hw/output/

Found 4 items
-rw-r--r--   1 hadoop users          0 2026-09-28 19:09 /hw1/wordcount_hw/output/_SUCCESS
-rw-r--r--   1 hadoop users      95021 2026-09-28 19:09 /hw1/wordcount_hw/output/part-00000
-rw-r--r--   1 hadoop users      95349 2026-09-28 19:09 /hw1/wordcount_hw/output/part-00001
-rw-r--r--   1 hadoop users      94585 2026-09-28 19:09 /hw1/wordcount_hw/output/part-00002


Посмотрим результаты выполнения job'ы

In [32]:
! hdfs dfs -cat /hw1/wordcount_hw/output/part-* | sort -k2,2nr | head -20

shall	9658
which	4119
their	3801
there	1799
before	1701
against	1587
shalt	1586
children	1559
said,	1556
them,	1549
LORD,	1308
saying,	1225
house	1221
people	1194
thee,	1170
every	1150
saith	1135
after	1053
Israel	1025
because	985
sort: write failed: 'standard output': Broken pipe
sort: write error


Создадим директорию для второго задания

In [33]:
! hdfs dfs -mkdir /hw1/top_5_sites
! hdfs dfs -mkdir -p /hw1/top_5_sites/input
! hdfs dfs -put ./Poseschenia_sai_774_tov.csv /hw1/top_5_sites/input/
! hdfs dfs -du -h /hw1/top_5_sites/input/

34.8 M  34.8 M  /hw1/top_5_sites/input/Poseschenia_sai_774_tov.csv


In [34]:
%%time
%%bash
# Запуск Hadoop Streaming для топ-5 сайтов:
# stream.num.map.output.key.fields=2 — ключ из двух частей (дата и url)
# map.output.key.field.separator='+' — части ключа разделены символом +
# keypartitioner.options='-k1,1' — строки с одной датой идут на один reducer
# KeyFieldBasedComparator + '-k1,1 -k2,2' — сортировка по дате, затем по url
hadoop jar /opt/hadoop/share/hadoop/tools/lib/hadoop-streaming-3.4.1.jar \
-D mapreduce.job.name="top5_sites" \
-D mapreduce.job.reduces=3 \
-D stream.num.map.output.key.fields=2 \
-D map.output.key.field.separator='+' \
-D mapreduce.partition.keypartitioner.options='-k1,1' \
-D mapreduce.job.output.key.comparator.class=org.apache.hadoop.mapreduce.lib.partition.KeyFieldBasedComparator \
-D mapreduce.partition.keycomparator.options='-k1,1 -k2,2' \
-files top_5_sites.py \
-mapper "python3 top_5_sites.py mapper" \
-reducer "python3 top_5_sites.py reducer" \
-input /hw1/top_5_sites/input/ \
-output /hw1/top_5_sites/output/ \
-partitioner org.apache.hadoop.mapred.lib.KeyFieldBasedPartitioner

packageJobJar: [/tmp/hadoop-unjar2203790577708461934/] [] /tmp/streamjob3342306034772527468.jar tmpDir=null
2026-09-28 19:11:40,817 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.19.0.4:8032
2026-09-28 19:11:40,900 INFO  [main] client.DefaultNoHARMFailoverProxyProvider (DefaultNoHARMFailoverProxyProvider.java:init(64)) - Connecting to ResourceManager at resourcemanager/172.19.0.4:8032
2026-09-28 19:11:41,040 INFO  [main] mapreduce.JobResourceUploader (JobResourceUploader.java:disableErasureCodingForPath(907)) - Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1790621459909_0004
2026-09-28 19:11:41,236 INFO  [main] mapred.FileInputFormat (FileInputFormat.java:listStatus(267)) - Total input files to process : 1
2026-09-28 19:11:41,279 INFO  [main] mapreduce.JobSubmitter (JobSubmitter.java:submitJobInternal(203)) - number of splits:2
2026-09-28 1

Проверим выполнение второго задания

In [35]:
! hdfs dfs -ls /hw1/top_5_sites/output/
! hdfs dfs -cat /hw1/top_5_sites/output/part-* | sort -k1,1 -k3,3nr | head -50

Found 4 items
-rw-r--r--   1 hadoop users          0 2026-09-28 19:11 /hw1/top_5_sites/output/_SUCCESS
-rw-r--r--   1 hadoop users        551 2026-09-28 19:11 /hw1/top_5_sites/output/part-00000
-rw-r--r--   1 hadoop users        376 2026-09-28 19:11 /hw1/top_5_sites/output/part-00001
-rw-r--r--   1 hadoop users        563 2026-09-28 19:11 /hw1/top_5_sites/output/part-00002
2024-05-26	https://gonzales-bautista.com/	335
2024-05-26	http://smith.com/	235
2024-05-26	https://www.smith.com/	221
2024-05-26	https://smith.com/	212
2024-05-26	http://www.smith.com/	212
2024-05-27	https://gonzales-bautista.com/	376
2024-05-27	https://www.smith.com/	270
2024-05-27	https://smith.com/	236
2024-05-27	http://smith.com/	215
2024-05-27	http://www.smith.com/	208
2024-05-28	https://gonzales-bautista.com/	368
2024-05-28	https://smith.com/	256
2024-05-28	https://www.smith.com/	251
2024-05-28	http://smith.com/	224
2024-05-28	http://www.smith.com/	204
2024-05-29	https://gonzales-bautista.com/	402
2024-05-29	htt